In [1]:
import numpy as np
from sklearn import datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, KFold
import warnings
warnings.filterwarnings('ignore')

1. Загрузите объекты из новостного датасета 20 newsgroups, относя
щиеся к категориям "космос"и "атеизм"(инструкция приведена вы
ше)

In [2]:
newsgroups = datasets . fetch_20newsgroups(
subset='all' ,
categories=[ 'alt.atheism' , 'sci.space' ])
X = newsgroups.data
y = newsgroups.target

2. Вычислите TF-IDF-признаки для всех текстов. Обратите внима
ние, что в этом задании мы предлагаем вам вычислить TF-IDF по
всем данным. При таком подходе получается, что признаки на обу
чающем множестве используют информацию из тестовой выборки
3
но такая ситуация вполне законна, поскольку мы не использу
ем значения целевой переменной из теста. На практике нередко
встречаются ситуации, когда признаки объектов тестовой выборки
известны на момент обучения, и поэтому можно ими пользоваться
при обучении алгоритма

In [3]:
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(X)

3. Подберите минимальныйлучшийпараметрCизмножества [10−5,10−4,...104,105]
для SVM с линейным ядром (kernel=’linear’) при помощи кросс
валидации по 5 блокам. Укажите параметр random_state=241 и
для SVM, и для KFold. В качестве меры качества используйте до
лю верных ответов (accuracy).

In [4]:
grid = {'C': np.power(10.0, np.arange(-3, 4))}
cv = KFold(n_splits=5, shuffle = True, random_state = 241)
clf = SVC(kernel = 'linear', random_state = 241)
gs = GridSearchCV(clf, grid, scoring = 'accuracy', cv = cv, n_jobs=-1)
gs.fit(X_tfidf, y)

print("\nРезультаты кросс-валидации:")
for score, params in zip(gs.cv_results_['mean_test_score'], 
                          gs.cv_results_['params']):
    print(f"C = {params['C']:8.0f}, accuracy = {score:.4f}")


Результаты кросс-валидации:
C =        0, accuracy = 0.5526
C =        0, accuracy = 0.5526
C =        0, accuracy = 0.9502
C =        1, accuracy = 0.9933
C =       10, accuracy = 0.9933
C =      100, accuracy = 0.9933
C =     1000, accuracy = 0.9933


4. Обучите SVMповсейвыборке с лучшимпараметром C, найденным
на предыдущем шаге.

In [8]:
best_C = gs.best_params_['C']
best_score = gs.best_score_
svm_best = SVC(kernel='linear', C=best_C, random_state=241)
svm_best.fit(X_tfidf, y)

coefficients = svm_best.coef_.toarray()[0]
feature_names = vectorizer.get_feature_names_out()
word_weights = list(zip(feature_names, coefficients))
word_weights_sorted = sorted(word_weights, key=lambda x: abs(x[1]), reverse=True)
top_words = [word for word, weight in word_weights_sorted[:10]]

5. Найдите 10 слов с наибольшим по модулю весом. Они являются
ответом на это задание. Укажите их через запятую, в нижнем ре
гистре, в лексикографическом порядке.

In [9]:
print(f"\n10 слов с наибольшим весом:")
for word, weight in word_weights_sorted[:10]:
    print(f"  {word}: {weight:.6f}")


10 слов с наибольшим весом:
  space: 2.663165
  god: -1.920379
  atheism: -1.254690
  atheists: -1.249180
  moon: 1.201611
  sky: 1.180132
  religion: -1.139081
  bible: -1.130612
  keith: -1.097094
  sci: 1.029307
